# Iceberg Basics — 10: Metadata Tables


Iceberg exposes rich metadata through queryable virtual tables.
These are invaluable for monitoring, debugging, and capacity planning.

Topics: .snapshots, .files, .manifests, .history, .partitions, .refs


In [ ]:
import os, time, pathlib, shutil, random, datetime
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *

MASTER   = 'spark://spark-master:7077'
DATA_DIR = '/workspace/data/iceberg_basics'
pathlib.Path(DATA_DIR).mkdir(parents=True, exist_ok=True)

spark = (
    SparkSession.builder
    .appName('iceberg-basics')
    .master(MASTER)
    .config('spark.executor.memory', '2g')
    .config('spark.driver.memory',   '1g')
    .config('spark.executor.cores',  '2')
    .config('spark.shuffle.sort.bypassMergeThreshold', '0')
    .getOrCreate()
)
spark.sparkContext.setLogLevel('WARN')
spark.sql("CREATE DATABASE IF NOT EXISTS local.icedb")
print(f"Spark {spark.version} | Iceberg catalog ready")

random.seed(42)
N = 100_000
CATEGORIES = ["Electronics","Clothing","Books","Food","Sports","Furniture"]
COUNTRIES  = ["US","UK","DE","FR","JP","CA","AU","BR"]
STATUSES   = ["pending","confirmed","shipped","delivered","cancelled"]

schema = StructType([
    StructField("order_id",    LongType()),
    StructField("customer_id", LongType()),
    StructField("product",     StringType()),
    StructField("category",    StringType()),
    StructField("country",     StringType()),
    StructField("quantity",    IntegerType()),
    StructField("price",       DoubleType()),
    StructField("revenue",     DoubleType()),
    StructField("status",      StringType()),
    StructField("order_date",  DateType()),
])

base = datetime.date(2024, 1, 1)
rows = []
for i in range(N):
    qty   = random.randint(1, 10)
    price = round(random.uniform(5, 1000), 2)
    d     = base + datetime.timedelta(days=random.randint(0, 364))
    rows.append((i+1, random.randint(1,10000),
                 f"Product_{random.randint(1,200)}",
                 random.choice(CATEGORIES), random.choice(COUNTRIES),
                 qty, price, round(qty*price, 2),
                 random.choice(STATUSES), d))

df = spark.createDataFrame(rows, schema)
df.cache().count()
print(f"Dataset ready: {N:,} rows")

## Step 1 — Setup



In [ ]:

spark.sql("DROP TABLE IF EXISTS local.icedb.meta_orders")
df.writeTo("local.icedb.meta_orders") \
  .partitionedBy("category") \
  .using("iceberg") \
  .createOrReplace()

# Add some history
df.limit(2000).writeTo("local.icedb.meta_orders").append()
spark.sql("UPDATE local.icedb.meta_orders SET status='shipped' WHERE status='confirmed' LIMIT 200")
spark.sql("""ALTER TABLE local.icedb.meta_orders CREATE TAG v1_stable RETAIN 30 DAYS""")
print("Table ready with history, partitions, and a tag")


## Step 2 — .snapshots and .history

Complete audit trail.

In [ ]:

print(".snapshots — all commits:")
spark.sql("""
    SELECT snapshot_id, committed_at, operation,
           summary['added-records']   AS added,
           summary['deleted-records'] AS deleted,
           summary['total-records']   AS total
    FROM local.icedb.meta_orders.snapshots
    ORDER BY committed_at
""").show(truncate=False)

print(".history — snapshot ancestry:")
spark.sql("""
    SELECT made_current_at, snapshot_id, is_current_ancestor
    FROM local.icedb.meta_orders.history
    ORDER BY made_current_at
""").show(truncate=False)


## Step 3 — .files and .manifests

Physical file inventory.

In [ ]:

print(".files — data file inventory:")
spark.sql("""
    SELECT content, COUNT(*) AS files,
           SUM(record_count) AS records,
           SUM(file_size_in_bytes)/1024/1024 AS total_mb,
           AVG(file_size_in_bytes)/1024 AS avg_kb
    FROM local.icedb.meta_orders.files
    GROUP BY content
""").show()

print(".manifests — manifest file inventory:")
spark.sql("""
    SELECT partition_spec_id,
           added_data_files_count,
           existing_data_files_count,
           deleted_data_files_count,
           length/1024 AS size_kb
    FROM local.icedb.meta_orders.manifests
    LIMIT 5
""").show()


## Step 4 — .partitions and .refs

Partition stats and named references.

In [ ]:

print(".partitions — per-partition statistics:")
spark.sql("""
    SELECT spec_id, partition.category AS category,
           record_count, file_count,
           total_data_file_size_in_bytes/1024 AS size_kb
    FROM local.icedb.meta_orders.partitions
    ORDER BY record_count DESC
""").show()

print(".refs — branches and tags:")
spark.sql("""
    SELECT name, type, snapshot_id,
           max_reference_age_in_ms/1000/60/60/24 AS max_age_days
    FROM local.icedb.meta_orders.refs
""").show(truncate=False)


## Step 5 — Building a Health Dashboard



In [ ]:

print("=== Iceberg Table Health Dashboard ===")
print()
detail_row = spark.sql("""
    SELECT snapshot_id, committed_at, operation
    FROM local.icedb.meta_orders.snapshots
    ORDER BY committed_at DESC LIMIT 1
""").collect()[0]

files_row = spark.sql("""
    SELECT COUNT(*) AS files,
           SUM(record_count) AS records,
           SUM(file_size_in_bytes)/1024/1024 AS total_mb,
           AVG(file_size_in_bytes)/1024 AS avg_kb
    FROM local.icedb.meta_orders.files
    WHERE content = 0
""").collect()[0]

snap_count = spark.sql("SELECT COUNT(*) FROM local.icedb.meta_orders.snapshots").collect()[0][0]

print(f"  Rows          : {files_row['records']:,}")
print(f"  Data files    : {files_row['files']}")
print(f"  Total size    : {files_row['total_mb']:.1f} MB")
print(f"  Avg file size : {files_row['avg_kb']:.0f} KB")
print(f"  Snapshots     : {snap_count}")
print(f"  Last op       : {detail_row['operation']} at {detail_row['committed_at']}")
print()
avg_kb = files_row['avg_kb'] or 0
if avg_kb < 1024:
    print(f"  ⚠️  Small files ({avg_kb:.0f} KB avg) — run rewrite_data_files")
else:
    print(f"  ✅ File sizes healthy ({avg_kb/1024:.1f} MB avg)")
if snap_count > 20:
    print(f"  ⚠️  Many snapshots ({snap_count}) — run expire_snapshots")
else:
    print(f"  ✅ Snapshot count healthy ({snap_count})")


## Summary: All Metadata Tables



In [ ]:

print("""
Iceberg metadata tables reference:

  table.snapshots    — all commits (snapshot_id, operation, summary stats)
  table.history      — snapshot ancestry (is_current_ancestor)
  table.files        — data file inventory (path, size, records, partition)
  table.manifests    — manifest file list (spec_id, file counts)
  table.partitions   — per-partition stats (record_count, file_count, size)
  table.refs         — branches and tags (name, type, snapshot_id)
  table.changes      — row-level change log (if CDF enabled)

Usage:
  SELECT * FROM local.db.table.snapshots
  SELECT * FROM local.db.table.files WHERE content = 0  -- 0=data, 1=pos_deletes, 2=eq_deletes
  SELECT * FROM local.db.table.partitions ORDER BY record_count DESC

Monitoring queries:
  -- Check for small files
  SELECT AVG(file_size_in_bytes)/1024/1024 AS avg_mb FROM table.files WHERE content=0

  -- Check snapshot accumulation
  SELECT COUNT(*) AS snap_count FROM table.snapshots

  -- Find largest partitions
  SELECT partition, record_count FROM table.partitions ORDER BY record_count DESC
""")
